# Demo: Kernel Mean Embeddings
presented at the International Conference on Probabilistic Numerics, Sophia Antipolis, France, September 2025

### Goal
We're trying to compute the kernel mean
$$k_P (x) = \int_{\Omega} k(x, x')\ \mathrm{d}P(x'),$$

### Purposes of the library
- for testing, prototyping
- easy to use library for getting kernel means
- easy to access code: all kernel means in one module, you can just copy it (MIT License).
- formulae documented in `README`

### Note
The library doesn't implement any method, but provides functionality for yours.

In [ ]:
import numpy as np

There are two ways to get your kernel mean

1. Set up your embedding from a kernel and a measure
2. Get your embedding directly

## 1. Set up your embedding from a kernel and a measure

### A note on dimensionality
No multi-dimensional embeddings are implemented right now in `kernel_embedding_dictionary`.  
In multi-dimensional problems, each dimension is modelled independently.


In [ ]:
ndim = 3  # Let's look at a three-dimensional example

#### Available Integration measures

| Measure Name | Object |
|-----|-----|
| Lebesgue Measure | `LebesgueMeasure` |
| Gaussian Measure | `GaussianMeasure` |

Again, every measure is configured through a config.

In [ ]:
from kernel_embedding_dictionary.measures import GaussianMeasure

In [ ]:
gaussian_measure_config = {
    "ndim": ndim,
    "means": [0.1, 0.2, 0.3],
    "variances": [1.3, 0.8, 1.1],
}

In [ ]:
my_gaussian = GaussianMeasure(gaussian_measure_config)

In [ ]:
# print the measure
print(my_gaussian)

### Available Kernels

| Kernel Name | Object | Works with measure |
|-----|-----|-----|
| Exponential Quadratic | `ExpQuadKernel` | Lebesgue, Gaussian | 
| General Matérn | `MaternKernel` | Lebesgue | 
| Matérn 1/2 | `Matern12Kernel` |  Lebesgue, Gaussian | 
| Matérn 3/2 | `Matern32Kernel` |  Lebesgue, Gaussian | 
| Matérn 5/2 | `Matern52Kernel` |  Lebesgue | 
| Matérn 7/2 | `Matern72Kernel` |  Lebesgue | 
| Wendland 0 | `Wendland0Kernel` |  Lebesgue, Gaussian | 
| Wendland 2 | `Wendland2Kernel` |  Gaussian | 

Each kernel is configured by a config, consult the `README` for parameters

In [ ]:
from kernel_embedding_dictionary.kernels import Matern12Kernel

In [ ]:
matern_config = {
    "ndim": ndim,
    "lengthscales": [1.2, 1.3, 1.4],
}

In [ ]:
my_matern = Matern12Kernel(matern_config)

In [ ]:
# print the kernel
print(my_matern)

### Initialize the kernel embedding with the kernel and measure

In [ ]:
from kernel_embedding_dictionary.embeddings import KernelEmbedding

In [ ]:
my_ke = KernelEmbedding(my_matern, my_gaussian)

In [ ]:
print(my_ke)

Note: the kernel embedding will through an error if the dimensions of kernel and measure are not identical.

In [ ]:
wrong_dim_config = {
    "ndim": 1,
    "lengthscale": 1.,
}

my_wrong_dim_matern = Matern12Kernel(wrong_dim_config)

wrong_ke = KernelEmbedding(my_wrong_dim_matern, my_gaussian)

### Compute the kernel mean

The kernel mean, defined as
$$k_P (x) = \int_{\Omega} k(x, x')\ \mathrm{d}P(x'),$$

can be computed at location $x$ with the `mean(x)` method of the kernel embedding.

In [ ]:
ndata = 10
x = np.random.randn(10, ndim)

my_ke.mean(x)

Again, a dimensionality mismatch will raise an error

In [ ]:
ndata = 10
x = np.random.randn(10, 4)

my_ke.mean(x)

## 2. Get your embedding directly

With the helper function `get_embedding`, you don't have to go the detour of defining your kernel and measure manually.

In [ ]:
from kernel_embedding_dictionary import get_embedding

In [ ]:
# Configs
expquad_config = {
    "ndim": ndim,
    "lengthscales": [1.0, 1.1, 0.5],
}

lebesgue_measure_config = {
    "ndim": ndim,
    "bounds": [(-2, 2), (-2, 2), (-1, 3)],
    "normalize": True
}

with the configs, just call `get_embedding`

In [ ]:
my_ke = get_embedding(kernel_name="expquad", measure_name="lebesgue", kernel_config=expquad_config, measure_config=lebesgue_measure_config)

In [ ]:
print(my_ke)

and the kernel mean:

In [ ]:
ndata = 10
x = np.random.randn(10, ndim)

my_ke.mean(x)

### Default setup

In fact, you don't even need to provide the config. Default values are assumed in that case, e.g.,
- `ndim` = 1
- `lengthscale` = 1.
- `mean` = 0.
- `variance` = 1.

In [ ]:
my_ke = get_embedding(kernel_name="wendland0", measure_name="gaussian")
print(my_ke)

### Some parameters won't work
Some kernel embeddings are available for default values only, because we could not find a closed form expression for non-default measures.

Also watch out that the parameters are _always_ given as a list, and specified in plural, i.e. `lengthscales`, not `lengthscale`.

In [ ]:
kernel_config = {
    "ndim": 1,
    "lengthscales": [0.5],
}

off_centered_gaussian_config = {
    "ndim": 1,
    "means": [0.5],
    "variances": [2.],
}

unavailable_ke = get_embedding(kernel_name="wendland2", measure_name="gaussian", kernel_config=kernel_config, measure_config=off_centered_gaussian_config)

# Let's make sure it's initialized correctly
print(unavailable_ke)

# Let's evaluate the kernel mean at random points
unavailable_ke.mean(np.random.randn(3,1))

### Plot mean embeddings

In [ ]:
from matplotlib import pyplot as plt

In [ ]:
# for plotting

def plot_kmean(ax, kernel_embedding, kname, mname, x_min=0, x_max=1, **kwargs):
    """Plot a kernel mean."""
    x = np.linspace(x_min, x_max, 100).reshape(-1, 1)
    ax.plot(x, kernel_embedding.mean(x), label="-".join([kname, mname]))

In [ ]:
all_kernels = [
    "expquad",
    "matern",
    "matern12",
    "matern32",
    "matern52",
    "matern72",
    "wendland0",
    "wendland2",
]

In [ ]:
# Lebesgue measure
measure = "lebesgue"
k_lengthscale = 0.1
kernel_config = {"lengthscales": [k_lengthscale]}

fig, ax = plt.subplots(1, 1, figsize=(10,4))

for kernel in all_kernels:
    try:
        ke = get_embedding(kernel, measure, kernel_config)
        plot_kmean(ax, ke, kernel, measure, x_min=0., x_max=1)
    except ValueError:
        continue

ax.legend(ncols=3)

In [ ]:
measure = "gaussian"
k_lengthscale = 10.
kernel_config = {"lengthscales": [k_lengthscale]}

fig, ax = plt.subplots(1, 1, figsize=(10,4))

for kernel in all_kernels:
    try:
        ke = get_embedding(kernel, measure, kernel_config)
        plot_kmean(ax, ke, kernel, measure, x_min=-10., x_max=10.)
    except ValueError:
        continue

ax.legend(ncols=3)

### Final notes

* All 1d kernel means have been numerically tested!
* WIP: integrated kernel means

__If you find a kernel mean that's not included, please contribute!__
https://github.com/mmahsereci/kernel_embedding_dictionary/